# Joshi Part 7: A Simple Monte Carlo Pricer

Based on **"The Concepts and Practice of Mathematical Finance"** by Mark S. Joshi, Part 7.

This notebook implements Joshi's SimpleMC1 through SimpleMC3, progressively building
a Monte Carlo pricer for European options using the RustQuant Python bindings.

## Key Concepts

Under risk-neutral pricing, the price of a European call is:

$$C = e^{-rT} \, \mathbb{E}[\max(S_T - K, 0)]$$

where $S_T$ follows Geometric Brownian Motion:

$$S_T = S_0 \exp\left((r - \tfrac{1}{2}\sigma^2)T + \sigma\sqrt{T}\,Z\right), \quad Z \sim N(0,1)$$

In [ ]:
import math
import random

# Parameters (same as Joshi's examples)
spot = 100.0
strike = 100.0
rate = 0.05
vol = 0.20
T = 1.0
n_paths = 100_000

## SimpleMC1: Raw Monte Carlo from Scratch

Joshi's first example is the simplest possible MC pricer.
We sample terminal values directly from the GBM distribution.

In [ ]:
random.seed(42)
payoff_sum = 0.0

for _ in range(n_paths):
    # Box-Muller transform (as Joshi describes)
    u1 = random.random()
    u2 = random.random()
    z = math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)
    
    # GBM terminal value
    s_t = spot * math.exp((rate - 0.5 * vol**2) * T + vol * math.sqrt(T) * z)
    
    # Call payoff
    payoff_sum += max(s_t - strike, 0.0)

mc_price_raw = math.exp(-rate * T) * payoff_sum / n_paths
print(f"SimpleMC1 - Raw MC Call Price: {mc_price_raw:.4f}")

## SimpleMC2: Using RustQuant's GBM Simulation

Now we use RustQuant's optimized GBM simulation engine (written in Rust)
for path generation, but compute payoffs in Python.

In [ ]:
from RustQuant.stochastics import GeometricBrownianMotion

gbm = GeometricBrownianMotion(mu=rate, sigma=vol)
traj = gbm.simulate(x0=spot, t_end=T, n_steps=1, n_paths=n_paths)

# Compute discounted payoffs
df = math.exp(-rate * T)
payoffs = [max(path[-1] - strike, 0.0) for path in traj.paths]
mc_price_gbm = df * sum(payoffs) / len(payoffs)

print(f"SimpleMC2 - RustQuant GBM Call Price: {mc_price_gbm:.4f}")

## SimpleMC3: Full RustQuant Pricing Engine

RustQuant's `BlackScholesMerton` class provides the exact analytic solution.
Let's compare all three approaches.

In [ ]:
from RustQuant.instruments import BlackScholesMerton, OptionType

# Analytic Black-Scholes price
call = BlackScholesMerton(
    underlying_price=spot,
    strike_price=strike,
    volatility=vol,
    risk_free_rate=rate,
    cost_of_carry=rate,
    expiry_year=2027, expiry_month=3, expiry_day=22,
    option_type=OptionType.Call,
)
put = BlackScholesMerton(
    underlying_price=spot,
    strike_price=strike,
    volatility=vol,
    risk_free_rate=rate,
    cost_of_carry=rate,
    expiry_year=2027, expiry_month=3, expiry_day=22,
    option_type=OptionType.Put,
)

print(f"{'Method':<30} {'Call':>10} {'Put':>10}")
print("-" * 50)
print(f"{'Raw MC (SimpleMC1)':<30} {mc_price_raw:>10.4f} {'N/A':>10}")
print(f"{'RustQuant GBM MC (SimpleMC2)':<30} {mc_price_gbm:>10.4f} {'N/A':>10}")
print(f"{'Black-Scholes (exact)':<30} {call.price():>10.4f} {put.price():>10.4f}")

## Greeks via Black-Scholes-Merton

Joshi emphasizes the importance of the Greeks. RustQuant provides all of them analytically.

In [ ]:
greeks = call.greeks()

print("BSM Call Greeks:")
for name, value in greeks.items():
    print(f"  {name:<10} = {value:.6f}")

## Implied Volatility Roundtrip

Given a market price, recover the implied volatility (Joshi discusses this in later chapters).

In [ ]:
market_price = call.price()
iv = call.implied_volatility(market_price)
print(f"Market price: {market_price:.6f}")
print(f"Implied vol:  {iv:.6f} (input was {vol})")

## Strike Sensitivity

How do prices and Greeks change across strikes? This is fundamental to understanding the volatility smile.

In [ ]:
print(f"{'Strike':>8} {'Price':>10} {'Delta':>10} {'Gamma':>10} {'Vega':>10}")
print("-" * 48)

for k in range(80, 125, 5):
    opt = BlackScholesMerton(
        underlying_price=spot, strike_price=float(k),
        volatility=vol, risk_free_rate=rate, cost_of_carry=rate,
        expiry_year=2027, expiry_month=3, expiry_day=22,
        option_type=OptionType.Call,
    )
    print(f"{k:>8} {opt.price():>10.4f} {opt.delta():>10.4f} {opt.gamma():>10.6f} {opt.vega():>10.4f}")

## Summary

| Joshi Section | Concept | RustQuant Equivalent |
|---------------|---------|---------------------|
| SimpleMC1 | Raw MC with Box-Muller | Manual Python implementation |
| SimpleMC2 | Parameterized MC | `GeometricBrownianMotion.simulate()` |
| SimpleMC3 | PayOff classes | `BlackScholesMerton` analytic pricing |

**Key takeaway**: Monte Carlo converges to the exact Black-Scholes price as $N \to \infty$,
with error proportional to $1/\sqrt{N}$.